# Stage II (vessels) - 07. Held-out test metrics

This notebook reproduces the vessel row of the Stage II results table. Steps:

1. **Configuration** - paths.
2. **Test split** - the 30 test images written by notebook 06.
3. **Deployed model** - `best_dice.pth` (EMA weights) and the threshold of the best validation
   epoch (0.46).
4. **FOV-aware evaluation** - flip test-time augmentation (identity, horizontal, vertical, both),
   prediction = probability > threshold, all counts inside the FOV, no post-processing.
5. **Results** - pixel-pooled Dice, IoU, sensitivity, specificity, precision, ROC AUC and PR AUC,
   and the per-image Dice (mean +/- SD, range, images with Dice >= 0.70).

## Step 1. Configuration

In [ ]:
from pathlib import Path

# Same layout as notebook 06.
DATA_ROOT = Path("data/vessel_dataset")
IMG_DIR = DATA_ROOT / "image"
MASK_DIR = DATA_ROOT / "mask"
SPLIT_CSV = Path("outputs/stage2/vessel/vessel_split.csv")   # from notebook 06
BEST_CKPT = Path("checkpoints/stage2/swinhrunetpp/best_dice.pth")
RESULTS_DIR = Path("outputs/stage2/vessel")                  # metric CSV / JSON written here

USE_TTA = True

In [ ]:
import json

import pandas as pd
import torch
from torch.utils.data import DataLoader

from hierarchiretina.stage2 import vessel_data as vd
from hierarchiretina.stage2.engine_vessel import load_best_model_and_threshold
from hierarchiretina.stage2.swinhrunetpp import IMG_SIZE, build_swinhrunetpp
from hierarchiretina.stage2.vessel_metrics import (evaluate_test_set, paper_table_row,
                                                   summarise_test_results)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## Step 2. Test split

The split CSV of notebook 06 is used when present; otherwise the split is re-created with the
same seed.

In [ ]:
if SPLIT_CSV.exists():
    _, _, test_pairs = vd.load_split_csv(SPLIT_CSV, IMG_DIR, MASK_DIR)
else:
    matched, _, _ = vd.load_dataset_files(IMG_DIR, MASK_DIR)
    _, _, test_pairs = vd.split_pairs(matched)
test_ds = vd.RetinalVesselDataset(test_pairs, IMG_SIZE, vd.get_val_transform(), crop_retina=True)
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False, num_workers=0)
print(f"Test images: {len(test_ds)} (paper: 30)")

## Step 3. Deployed model and threshold

In [ ]:
model = build_swinhrunetpp(pretrained=False, img_size=IMG_SIZE)
model, THR, info = load_best_model_and_threshold(BEST_CKPT, model)
model = model.to(DEVICE).eval()
info   # expected: weights 'ema', best_epoch 53, threshold ~0.46

## Step 4. FOV-aware evaluation

In [ ]:
per_image, pooled = evaluate_test_set(model, test_loader, DEVICE, THR, use_tta=USE_TTA)
per_image.insert(1, "image", [Path(p[0]).name for p in test_pairs])
summary = summarise_test_results(per_image, pooled, THR, USE_TTA)

per_image.to_csv(RESULTS_DIR / "test_metrics_per_image.csv", index=False)
(RESULTS_DIR / "test_metrics_summary.json").write_text(json.dumps(summary, indent=2))

## Step 5. Results

Paper (vessel row): Dice 0.731, IoU 0.576, sensitivity 0.915, precision 0.609, specificity
0.905, ROC AUC 0.966, PR AUC 0.852; per-image Dice 0.727 +/- 0.043 (range 0.615-0.770,
25 of 30 images >= 0.70).

In [ ]:
paper_table_row(summary).round(3)

In [ ]:
metrics = ["dice", "iou", "sensitivity", "specificity", "precision", "f1", "accuracy"]
table = pd.DataFrame({
    "image-averaged (mean)": [summary["image_avg"][m][0] for m in metrics],
    "image-averaged (SD)": [summary["image_avg"][m][1] for m in metrics],
    "pixel-pooled": [summary["pixel_pooled"][m] for m in metrics],
}, index=metrics).round(4)
print(f"Per-image Dice range: {summary['image_dice_range'][0]:.3f}-"
      f"{summary['image_dice_range'][1]:.3f} | images with Dice >= 0.70: "
      f"{summary['n_image_dice_ge_0p70']} of {summary['n_test_images']}")
table